# Step 2 — Indexing: Incidence Matrix dan Inverted Index

**Tujuan praktikum**
1. Mahasiswa memahami representasi *term-document incidence matrix* dan kelemahannya.
2. Mahasiswa mampu membangun *inverted index* (kamus + *posting list*) beserta *document frequency* dan *term frequency*.
3. Mahasiswa mampu membangun *positional index* dan menggunakannya untuk *phrase query*.

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from collections import Counter, defaultdict
from stki_utils import corpus_texts, preprocess

teks = corpus_texts()                      # {doc_id: judul + isi}
docs = {d: preprocess(t) for d, t in teks.items()}
print("Jumlah dokumen:", len(docs))
print("D07:", docs["D07"])

Jumlah dokumen: 30
D07: ['budidaya', 'padi', 'sawah', 'budidaya', 'padi', 'sawah', 'butuh', 'air', 'cukup', 'pupu', 'tepat', 'tani', 'anjur', 'guna', 'benih', 'unggul', 'hasil', 'panen', 'tingkat', 'hama', 'wereng', 'cokelat', 'jadi', 'ancam', 'utama', 'tanam', 'padi']


## Langkah 1 — Term-document incidence matrix
Matriks biner berukuran |V| × N: bernilai 1 jika term muncul di dokumen. Kita coba untuk 6 dokumen topik pertanian.

In [3]:
subset = ["D07", "D08", "D09", "D10", "D11", "D12"]
vocab_subset = sorted({t for d in subset for t in docs[d]})
incidence = pd.DataFrame([[int(t in docs[d]) for d in subset] for t in vocab_subset],
                         index=vocab_subset, columns=subset)
print("Ukuran matriks:", incidence.shape)
incidence.loc[["air", "hama", "padi", "pupuk", "sawah", "tani", "tanah"]]

Ukuran matriks: (109, 6)


,D07,D08,D09,D10,D11,D12
air,1,0,1,1,0,0
hama,1,0,0,0,1,0
padi,1,0,0,0,0,0
pupuk,0,1,0,1,0,0
sawah,1,0,0,0,1,0
tani,1,1,1,1,1,1
tanah,0,1,1,1,0,0


In [3]:
vocab = sorted({t for tokens in docs.values() for t in tokens})
sel_total = len(vocab) * len(docs)
sel_isi = sum(len(set(tokens)) for tokens in docs.values())
print(f"Kosakata: {len(vocab)} term, {len(docs)} dokumen")
print(f"Sel matriks penuh : {sel_total}")
print(f"Sel bernilai 1    : {sel_isi} ({sel_isi / sel_total:.1%})")

Kosakata: 472 term, 30 dokumen
Sel matriks penuh : 14160
Sel bernilai 1    : 702 (5.0%)


Sebagian besar sel bernilai 0 (*sparse*). Pada koleksi nyata berisi jutaan dokumen, matriks penuh tidak mungkin disimpan. Solusinya adalah **inverted index**: untuk setiap term hanya disimpan daftar dokumen yang memuatnya.

## Langkah 2 — Membangun inverted index
Struktur yang dibangun: `index[term] = {doc_id: tf}`. Panjang posting list sama dengan *document frequency* (df).

In [4]:
def build_inverted_index(docs_tokens):
    index = defaultdict(dict)
    for doc_id, tokens in docs_tokens.items():
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(sorted(index.items()))

index = build_inverted_index(docs)
print("Jumlah term di kamus:", len(index))
for term in ["tani", "digital", "gizi", "cari"]:
    print(f"{term:8s} df={len(index[term]):2d}  postings={index[term]}")

Jumlah term di kamus: 472
tani     df= 7  postings={'D07': 1, 'D08': 2, 'D09': 1, 'D10': 3, 'D11': 1, 'D12': 3, 'D23': 1}
digital  df= 5  postings={'D19': 1, 'D21': 3, 'D25': 1, 'D27': 3, 'D29': 3}
gizi     df= 2  postings={'D02': 2, 'D30': 3}
cari     df= 2  postings={'D13': 3, 'D29': 1}


In [5]:
tabel = pd.DataFrame({
    "df": {t: len(p) for t, p in index.items()},
    "cf (total tf)": {t: sum(p.values()) for t, p in index.items()},
}).sort_values("df", ascending=False)
tabel.head(10)

,df,cf (total tf)
guna,11,12
lalu,9,9
bantu,8,8
tani,7,12
data,6,9
ajar,6,11
tingkat,6,6
tinggi,6,7
jadi,6,7
digital,5,11


## Langkah 3 — Menyimpan dan memuat index

In [6]:
import json, os

os.makedirs("output", exist_ok=True)
with open("output/inverted_index.json", "w") as f:
    json.dump(index, f)

with open("output/inverted_index.json") as f:
    index_dimuat = json.load(f)
print("Index tersimpan dan dimuat ulang dengan benar:", index_dimuat == index)
print("Ukuran file:", os.path.getsize("output/inverted_index.json"), "byte")

Index tersimpan dan dimuat ulang dengan benar: True
Ukuran file: 12606 byte


## Langkah 4 — Positional index dan phrase query
*Positional index* menyimpan posisi kemunculan term: `index[term][doc_id] = [posisi, ...]`. Dengan posisi, kita bisa mencari frasa, yaitu term yang muncul berurutan.

In [8]:
def build_positional_index(docs_tokens):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in docs_tokens.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {t: dict(p) for t, p in sorted(index.items())}

pos_index = build_positional_index(docs)
print("cerdas :", pos_index["cerdas"])
print("buat   :", pos_index["buat"])

cerdas : {'D10': [1, 5], 'D14': [0, 3], 'D27': [9]}
buat   : {'D08': [0, 5, 22], 'D14': [1, 4]}


In [9]:
def phrase_search(frasa, pos_index):
    terms = preprocess(frasa)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:22s} -> {phrase_search(frasa, pos_index)}")

'kecerdasan buatan'    -> ['D14']
'pertanian cerdas'     -> ['D10']
'pupuk organik'        -> ['D08']
'data pribadi'         -> ['D15']


## Tugas Step 2
1. Hitung perbandingan ukuran (jumlah entri) antara incidence matrix penuh dan inverted index untuk seluruh korpus. Berapa kali lebih hemat inverted index?
2. Tampilkan 10 term dengan df = 1 (*hapax*). Apakah term-term tersebut berguna untuk pencarian? Jelaskan.
3. Posisi pada positional index di atas dihitung **setelah** stopword dihapus. Apa akibatnya pada pencarian frasa `"sistem temu kembali informasi"` atau `"gerakan 3M"`? Modifikasi kode agar posisi dihitung sebelum stopword dihapus.
4. Buat fungsi `proximity_search(t1, t2, k)` yang mengembalikan dokumen yang memuat t1 dan t2 dengan jarak paling jauh k posisi.

In [10]:
# Tulis jawaban Anda di sini